# PyTorch Text Classification
This notebook demonstrates how to perform text classification using PyTorch and the BERT transformer model. The workflow includes dataset download, preprocessing, tokenization, model building, training, and evaluation.

## Setup
In this section, we set up the environment and download the required dataset from Kaggle.

In [1]:
# Set up Kaggle API credentials for downloading datasets
import os  # For file and directory operations
import shutil  # For file copying

# The Kaggle API key is required to download datasets from Kaggle. This code copies the API key to the default Kaggle directory and sets the correct permissions.
kaggle_json_path = './kaggle.json'  # Path to your Kaggle API key
kaggle_dir = os.path.expanduser('~/.kaggle')  # Default Kaggle directory
os.makedirs(kaggle_dir, exist_ok=True)  # Create directory if it doesn't exist
shutil.copy(kaggle_json_path, os.path.join(kaggle_dir, 'kaggle.json'))  # Copy API key
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)  # Set permissions
print('Kaggle API key set up successfully.')

Kaggle API key set up successfully.


In [2]:
# Install and import opendatasets, then download the news headlines dataset for sarcasm detection from Kaggle
!pip install -q opendatasets
import opendatasets as od
# Download the dataset using the Kaggle link. This will create a folder with the dataset files.
od.download("https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection")

Skipping, found downloaded files in ".\news-headlines-dataset-for-sarcasm-detection" (use force=True to force download)


## Imports
Import all necessary libraries for data processing, model building, and training.

In [3]:
# Import all required libraries for data processing, model building, and training
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import pandas as pd
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import numpy as np
from sklearn.model_selection import train_test_split

# Set a random seed for reproducibility
torch.manual_seed(42)

# Set device to GPU if available, else CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Is ROCm/CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")

c:\Users\lovep\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.1+rocm7.2.1
Is ROCm/CUDA available: True
Device Name: AMD Radeon RX 7900 XT


## Data Preparation and Exploration
Load the dataset, clean it, and prepare it for training.

## Model Training and Evaluation
In this section, you would typically define the training loop, loss function, optimizer, and evaluation metrics. You would then train the model and evaluate its performance on the validation and test sets.

### Training Loop Explanation
The following code block implements the training and validation loop for the text classification model:
- **Metric Tracking:** Lists are initialized to store loss and accuracy for both training and validation sets across epochs.
- **Epoch Loop:** For each epoch, the model iterates over the training data, computes predictions, calculates loss, updates model weights, and tracks accuracy.
- **Validation:** After each epoch, the model is evaluated on the validation set without updating weights to monitor generalization performance.
- **Loss and Accuracy Calculation:** For both training and validation, the code accumulates loss and counts correct predictions to compute average metrics per epoch.

In [4]:
# Load the sarcasm headlines dataset as a pandas DataFrame
data_df = pd.read_json('news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json',lines=True)
# Remove missing values and duplicates to ensure data quality
data_df.dropna(inplace=True)
data_df.drop_duplicates(inplace=True)
# Drop the 'article_link' column as it is not needed for classification
data_df.drop(columns=['article_link'], inplace=True)
print(data_df.shape)
# Display the first few rows of the cleaned dataset
data_df.head()

(26708, 2)


,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


In [5]:
# Split the data into training, validation, and test sets
X_train, X_test, Y_train, Y_test = train_test_split(np.array(data_df['headline']), np.array(data_df['is_sarcastic']), test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=0.5, random_state=42)

# Print the size of each split to verify the distribution
print(f"Training set size: {len(X_train)} (% {len(X_train)/len(data_df)*100:.2f}%)")
print(f"Validation set size: {len(X_val)} (% {len(X_val)/len(data_df)*100:.2f}%)")
print(f"Test set size: {len(X_test)} (% {len(X_test)/len(data_df)*100:.2f}%)")

Training set size: 18695 (% 70.00%)
Validation set size: 4006 (% 15.00%)
Test set size: 4007 (% 15.00%)


In [6]:
# Load the BERT tokenizer and model from Hugging Face Transformers
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1478.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Define a custom PyTorch Dataset for tokenized text data
class dataset(Dataset):
    def __init__(self,X,Y):
        # Tokenize each headline and move tensors to the selected device
        self.X = [tokenizer(x, 
                            padding='max_length', 
                            truncation=True, 
                            max_length=100, 
                            return_tensors='pt').to(device) 
                            for x in X
        ]
        self.Y = torch.tensor(Y, dtype=torch.float32).to(device)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Create dataset objects for training, validation, and test sets
training_data = dataset(X_train, Y_train)
validation_data = dataset(X_val, Y_val)
test_data = dataset(X_test, Y_test)

In [8]:
# Set hyperparameters for training
BATCH_SIZE = 32  # Number of samples per batch
LR = 1e-4        # Learning rate
EPOCHS = 10      # Number of training epochs

In [9]:
# Create DataLoader objects for efficient data batching and shuffling
train_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(validation_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [10]:
# Define the neural network model using BERT as a feature extractor
class MyModul(nn.Module):
    def __init__(self, bert_model):
        super(MyModul, self).__init__()
        
        self.bert = bert_model
        self.drop = nn.Dropout(0.25)  # Dropout layer for regular
        self.linear1= nn.Linear(768, 384)  # Output layer for binary classification
        self.linear2 = nn.Linear(384, 1)  # Output layer for binary classification
        self.sigmoid = nn.Sigmoid()  # Sigmoid activation for binary classification
        
    def forward(self, input_ids, attention_mask):
        # Use BERT to extract features from input text
        with torch.no_grad():
            pooled_outputs = self.bert(input_ids, attention_mask, return_dict=False)[0][:,0]  # Get the last hidden state
            output = self.linear1(pooled_outputs)
            output = self.drop(output)
            output = self.linear2(output)
            output = self.sigmoid(output)
        return output

In [11]:
for param in bert_model.parameters():
    param.requires_grad = False  # Freeze BERT parameters to prevent training
# Instantiate the model and move it to the selected device
model = MyModul(bert_model).to(device)

In [12]:
model

MyModul(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [13]:
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = Adam(model.parameters(), lr=LR)  # Adam optimizer for training the model

In [ ]:
total_loss_train_plot = []
total_loss_val_plot = []
total_acc_train_plot = []
total_acc_val_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_acc_val = 0
    total_loss_val = 0
    
    for indx, data in enumerate(train_loader):
        inputs, labels = data
        inputs.to(device)
        labels.to(device)
        
        prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
        batch_loss = criterion(prediction, labels)
        total_loss_train += batch_loss.item()
        
        acc = (prediction.round() == labels).sum().item()
        total_acc_train += acc
        
        optimizer.step()
        optimizer.zero_grad()
    
    with torch.no_grad():
        for indx, data in enumerate(val_loader):
            inputs, labels = data
            inputs.to(device)
            labels.to(device)
            
            prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
            batch_loss = criterion(prediction, labels)
            total_loss_val += batch_loss.item()
            
            acc = (prediction.round() == labels).sum().item()
            total_acc_val += acc
            
        total_loss_train_plot.append(round(total_loss_train/1000, 4))
        total_loss_val_plot.append(round(total_loss_val/1000, 4))
        total_acc_train_plot.append(round(total_acc_train/training_data.__len__() * 100, 4))
        total_acc_val_plot.append(round(total_acc_val/validation_data.__len__() * 100, 4))
    
print(f""" Epochs: {epoch+1}, 
      Training Loss: {total_loss_train/1000}, 
      Validation Loss: {total_loss_val/1000}, 
      Training Accuracy: {total_acc_train/training_data.__len__() * 100}, 
      Validation Accuracy: {total_acc_val/validation_data.__len__() * 100}
      """)

In [ ]:
with torch.no_grad():
    total_acc_test = 0
    total_loss_test = 0
    
    for indx, data in enumerate(test_loader):
        inputs, labels = data
        inputs.to(device)
        labels.to(device)
        
        prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
        batch_loss = criterion(prediction, labels)
        total_loss_test += batch_loss.item()
        
        acc = (prediction.round() == labels).sum().item()
        total_acc_test += acc

print(f""" Test Loss: {total_loss_test/test_data.__len__()}, 
      Test Accuracy: {total_acc_test/test_data.__len__() * 100}
      """)